# Video sampling logic

In [ ]:
import pickle
import torch
from pathlib import Path
import numpy as np
import random
import pandas as pd
from ultralytics import YOLO
from decord import VideoReader
from decord import cpu, gpu
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as T

In [32]:
def get_m_vids_from_novel_class(gttubes, samples_set, novel_classes_ids, M):
    # Get the videos that include those classes and sample at least K 
    video_dict = {} # video id and number of instances of the action in the class action : {video_id1: 4, video_id2: 4}

    # Now sample those classes in order (they are already random so it's fine if we don't mix again)
    query_videos = set()
    accumulated_counts = {cls: 0 for cls in novel_classes_ids}

    # Shuffle the videos
    vids_set = list(samples_set)
    random.shuffle(vids_set)

    for video in vids_set:
        # video_dict[novel_class_id] = {}
        # Check if we have enough videos for all classes
        if all(count >= M for count in accumulated_counts.values()):
            break

        # Get the tubes for the video
        tubes = gttubes[video] 
        
        # Check if the video has any classes we need
        needs_video = False
        for cls in novel_classes_ids:
            if cls in tubes and accumulated_counts[cls] < M:
                needs_video = True
                break
        
        # if needed, ad the video
        if needs_video:
            query_videos.add(video)
            for cls in novel_classes_ids:
                if cls in tubes:
                    accumulated_counts[cls] += len(tubes[cls])
                
    return list(query_videos), accumulated_counts
# Random order of classes - than start by sampling videos for first class - for the next ones already count the videos decided
# If there is too many for some class, just do not count those ones or something like that 
# You have to exclude them randomly


def load_video(path, F = 8):
    vr = VideoReader(path, ctx=cpu(0))
    frames =  vr.get_batch(range(len(vr)))
    del vr
    return frames.asnumpy()

# The actual evaluation

In [ ]:
project_root = Path.cwd().parent
data_root = str(project_root / "multisports_data/data/trainval")
ground_truth_path= str(project_root / "multisports_fewshot_GT.pkl")

# Read The GT file
with open(ground_truth_path, "rb") as file:
    data = pickle.load(file)

# GTtubes
gttubes = data["gttubes"]

# Get the test classes
possible_classes = data["test_labels"]

# Splitting logic to query and support 
# Pick N novel classes
N = 5
# Pick at least M samples from those classes
M = 15
# NUmber of support samples
K = 5

# Sample classes
novel_classes_ids = random.sample(range(len(possible_classes)), N)

# Test videos
test_videos = data["test_videos"]

# Evaluation loop
E = 1

# Here just use the pre-trained dinov2 model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
model.eval()
model = model.to(device)

transform = T.Compose([
    T.Resize((224, 224), antialias=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# Episodic evaluation
for episode in tqdm(range(E)):
    # Sample the query and support videos
    query_videos, _ = get_m_vids_from_novel_class(gttubes, test_videos, novel_classes_ids, M)
    remaining_vids = list(set(test_videos) - set(query_videos))
    support_videos, _ = get_m_vids_from_novel_class(gttubes, remaining_vids, novel_classes_ids, K)

    # Build prototypes
    prototypes = {id:[] for id in novel_classes_ids}
    for id in tqdm(novel_classes_ids):
        for vid in support_videos:
            path = data_root + f"/{vid}.mp4"
            print(path)

            # Get the frames from the video
            frames = load_video(path)
            frames_tensor = torch.from_numpy(frames).permute(0, 3, 1, 2).float()
            frames_tensor = frames_tensor / 255.0
            frames_tensor = transform(frames_tensor).to(device)

            # Get the feaures using the model
            with torch.no_grad():
                frame_features = model(frames_tensor)

            # Temporally average the features 
            video_prototype = frame_features.mean(dim=0)
            
            # Append the prototype for each video to the list
            prototypes[id].append(video_prototype.cpu())

        # Build the prototype as the average of features from support videos
        prototypes[id] = torch.stack(np.mean(prototypes[id])).mean(dim=0)        